In [ ]:
# FIGURE SCRIPT A — Episode PnL Distribution Plots
#
# Generates two figures:
#
# A1: Overlapping PnL histograms (full state, all folds pooled)
# A2: Full state vs reduced state PPO/DDPG comparison
#
# Requires: s1_full_results.json, s2_reduced_results.json
# Output:   fig_pnl_distribution_full.png
#           fig_pnl_distribution_comparison.png



import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import json, os

DRIVE_PATH = "/content/drive/MyDrive/MEng Project"

COLOURS = {
    "PPO":                     "#4A90D9",
    "DDPG":                    "#2ECC71",
    "Mean Reversion (Oracle)": "#F39C12",
    "Naive Hold":              "#E8453C",
}

#Load data

with open(f"{DRIVE_PATH}/s1_full_results.json") as f:
    s1 = json.load(f)

with open(f"{DRIVE_PATH}/s2_reduced_results.json") as f:
    s2 = json.load(f)

# Pool all episodes across folds

def pool(results, key):
    all_pnls = []
    for fold in results:
        all_pnls.extend(fold[f"{key}_pnls_list"])
    return np.array(all_pnls)

ppo_full  = pool(s1, "ppo")
ddpg_full = pool(s1, "ddpg")
mr_full   = pool(s1, "mr")
nh_full   = pool(s1, "nh")

ppo_red  = pool(s2, "ppo")
ddpg_red = pool(s2, "ddpg")

print(f"Full state episodes: PPO={len(ppo_full)}, DDPG={len(ddpg_full)}")
print(f"Reduced state episodes: PPO={len(ppo_red)}, DDPG={len(ddpg_red)}")

#Figure A1: Full state PnL distributions

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle(
    "Episode PnL Distributions — Full State Walk-Forward (All Folds Pooled)",
    fontsize=13,
    fontweight="bold"
)

agents = [
    ("PPO",                     ppo_full),
    ("DDPG",                    ddpg_full),
    ("Mean Reversion (Oracle)", mr_full),
    ("Naive Hold",              nh_full),
]

# Left: full range (clip extreme DDPG outliers for readability)

p1, p99 = np.percentile(np.concatenate([a for _, a in agents]), [0.5, 99.5])
bins = np.linspace(p1, p99, 80)

for label, pnls in agents:
    clipped = np.clip(pnls, p1, p99)
    axes[0].hist(
        clipped,
        bins=bins,
        alpha=0.55,
        color=COLOURS[label],
        label=label,
        density=True,
        linewidth=0.5,
        edgecolor="white"
    )
    axes[0].axvline(
        np.mean(pnls),
        color=COLOURS[label],
        linewidth=2,
        linestyle="--",
        alpha=0.9
    )

axes[0].axvline(0, color="black", linewidth=1.2, linestyle="-", alpha=0.5)
axes[0].set_xlabel("Episode PnL (EUR/MWh)", fontsize=11)
axes[0].set_ylabel("Density", fontsize=11)
axes[0].set_title("Full Distribution (0.5th–99.5th percentile)", fontsize=11)
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.2)

# Right: zoomed on PPO vs DDPG only — shows structure more clearly

p2, p98 = np.percentile(np.concatenate([ppo_full, ddpg_full]), [2, 98])
bins2 = np.linspace(p2, p98, 70)

for label, pnls in [("PPO", ppo_full), ("DDPG", ddpg_full)]:
    clipped = np.clip(pnls, p2, p98)
    axes[1].hist(
        clipped,
        bins=bins2,
        alpha=0.6,
        color=COLOURS[label],
        label=f"{label} (mean={np.mean(pnls):.2f}, Sharpe={np.mean(pnls)/(np.std(pnls)+1e-8):.3f})",
        density=True,
        linewidth=0.5,
        edgecolor="white"
    )
    axes[1].axvline(
        np.mean(pnls),
        color=COLOURS[label],
        linewidth=2.5,
        linestyle="--",
        alpha=0.95,
        label=f"_{label} mean"
    )

axes[1].axvline(0, color="black", linewidth=1.2, linestyle="-", alpha=0.5, label="Zero")
ppo_pos = (ppo_full > 0).mean() * 100
ddpg_pos = (ddpg_full > 0).mean() * 100

axes[1].set_xlabel("Episode PnL (EUR/MWh)", fontsize=11)
axes[1].set_ylabel("Density", fontsize=11)
axes[1].set_title(
    f"PPO vs DDPG  |  PPO: {ppo_pos:.1f}% positive  |  DDPG: {ddpg_pos:.1f}% positive",
    fontsize=10
)
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.2)

plt.tight_layout()

out1 = f"{DRIVE_PATH}/fig_pnl_distribution_full.png"
plt.savefig(out1, dpi=180, bbox_inches="tight")
plt.show()

print(f"Saved: {out1}")

#Figure A2: Full vs Reduced state comparison

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle(
    "Full State vs Reduced State — PPO and DDPG Episode PnL Distributions",
    fontsize=13,
    fontweight="bold"
)

# PPO comparison

p_lo = np.percentile(np.concatenate([ppo_full, ppo_red]), 1)
p_hi = np.percentile(np.concatenate([ppo_full, ppo_red]), 99)
bins3 = np.linspace(p_lo, p_hi, 70)

axes[0].hist(
    np.clip(ppo_full, p_lo, p_hi),
    bins=bins3,
    alpha=0.6,
    color="#4A90D9",
    label=f"Full State (Sharpe={np.mean(ppo_full)/(np.std(ppo_full)+1e-8):.3f})",
    density=True
)

axes[0].hist(
    np.clip(ppo_red, p_lo, p_hi),
    bins=bins3,
    alpha=0.5,
    color="#A0C4F1",
    label=f"Reduced State (Sharpe={np.mean(ppo_red)/(np.std(ppo_red)+1e-8):.3f})",
    density=True
)

axes[0].axvline(np.mean(ppo_full), color="#4A90D9", linewidth=2.5, linestyle="--")
axes[0].axvline(np.mean(ppo_red),  color="#A0C4F1", linewidth=2.5, linestyle="--")
axes[0].axvline(0, color="black", linewidth=1.0, linestyle="-", alpha=0.5)

axes[0].set_xlabel("Episode PnL (EUR/MWh)", fontsize=11)
axes[0].set_ylabel("Density", fontsize=11)
axes[0].set_title("PPO: Full State vs Reduced State", fontsize=11)
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.2)

# DDPG comparison

d_lo = np.percentile(np.concatenate([ddpg_full, ddpg_red]), 1)
d_hi = np.percentile(np.concatenate([ddpg_full, ddpg_red]), 99)
bins4 = np.linspace(d_lo, d_hi, 70)

axes[1].hist(
    np.clip(ddpg_full, d_lo, d_hi),
    bins=bins4,
    alpha=0.6,
    color="#2ECC71",
    label=f"Full State (Sharpe={np.mean(ddpg_full)/(np.std(ddpg_full)+1e-8):.3f})",
    density=True
)

axes[1].hist(
    np.clip(ddpg_red, d_lo, d_hi),
    bins=bins4,
    alpha=0.5,
    color="#A8E6C3",
    label=f"Reduced State (Sharpe={np.mean(ddpg_red)/(np.std(ddpg_red)+1e-8):.3f})",
    density=True
)

axes[1].axvline(np.mean(ddpg_full), color="#2ECC71", linewidth=2.5, linestyle="--")
axes[1].axvline(np.mean(ddpg_red),  color="#A8E6C3", linewidth=2.5, linestyle="--")
axes[1].axvline(0, color="black", linewidth=1.0, linestyle="-", alpha=0.5)

axes[1].set_xlabel("Episode PnL (EUR/MWh)", fontsize=11)
axes[1].set_ylabel("Density", fontsize=11)
axes[1].set_title("DDPG: Full State vs Reduced State", fontsize=11)
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.2)

plt.tight_layout()

out2 = f"{DRIVE_PATH}/fig_pnl_distribution_comparison.png"
plt.savefig(out2, dpi=180, bbox_inches="tight")
plt.show()

print(f"Saved: {out2}")
print("\nFigure A complete.")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/MEng Project/s1_full_results.json'